In [1]:
# Corridor géofer entre deux gares : suit la vraie ligne de chemin de fer
# (données OpenStreetMap, cf. openrailwaymap.org) entre les deux gares au
# lieu d'une approximation géométrique — une simple distance à une voie ne
# suffit pas (une gare peut être aussi proche d'une ligne différente), donc
# on construit un graphe du réseau ferré local et on calcule le plus court
# chemin réel entre les deux gares, puis on ne garde que les gares dans un
# petit tampon autour de ce chemin. Quand une agglomération a plusieurs
# gares sur ce chemin (ex. La Rochelle / La Rochelle Porte Dauphine), seule
# la plus fréquentée est retenue : une gare de corridor = une commune.
import os
import time

import geopandas as gpd
import networkx as nx
import pandas as pd
import requests
from shapely.geometry import LineString, Point
from shapely.ops import unary_union

import app

GEOFER_DIR = "Data_geofer"
OUTPUT_DIR = "Output"
GARE_DEPART = "Reims"
GARE_ARRIVEE = "Epernay"
MARGE_BBOX_DEG = 0.4  # marge autour des deux gares pour la requête Overpass
SEUIL_DISTANCE_ALERTE_KM = 200  # message d'erreur dédié au-delà, si aucune voie ne relie les deux gares
TAMPON_VOIE_M = 300  # distance max à la voie réelle pour qu'une gare soit retenue

OVERPASS_MIRRORS = [
    "https://maps.mail.ru/osm/tools/overpass/api/interpreter",
    "https://overpass.osm.ch/api/interpreter",
    "https://overpass-api.de/api/interpreter",
]

# Dossier de cache pour ce corridor, réutilisé par toutes les cellules
# suivantes (fichiers intermédiaires + résultats finaux).
nom_corridor = f"{GARE_DEPART}_{GARE_ARRIVEE}_corridor".replace(" ", "_")
dossier_corridor = os.path.join(OUTPUT_DIR, nom_corridor)
os.makedirs(dossier_corridor, exist_ok=True)


def requete_overpass(query: str, essais: int = 2) -> dict:
    headers = {"User-Agent": "corridor-analyse-fr/1.0 (contact: antoine.chevre@gmail.com)"}
    derniere_erreur = None
    for url in OVERPASS_MIRRORS:
        for _ in range(essais):
            try:
                r = requests.post(url, data={"data": query}, headers=headers, timeout=60)
                r.raise_for_status()
                return r.json()
            except Exception as exc:
                derniere_erreur = exc
                time.sleep(1)
    raise RuntimeError(f"Overpass indisponible sur tous les miroirs : {derniere_erreur}")


gares = pd.read_csv(f"{GEOFER_DIR}/geofer_gares.csv")
gares = gares[gares["siOuverte"]].copy()
gares["codeUic"] = gares["codeUic"].astype(str)
gares["inseeCommune"] = gares["inseeCommune"].astype(str).str.zfill(5)
gares["inseeDepartement"] = gares["inseeDepartement"].astype(str)
gares = gpd.GeoDataFrame(
    gares, geometry=gpd.points_from_xy(gares["wgs84Lon"], gares["wgs84Lat"]), crs="EPSG:4326",
)

ligne_depart = gares.loc[gares["nomGare"] == GARE_DEPART].iloc[0]
ligne_arrivee = gares.loc[gares["nomGare"] == GARE_ARRIVEE].iloc[0]
sud = min(ligne_depart["wgs84Lat"], ligne_arrivee["wgs84Lat"]) - MARGE_BBOX_DEG
nord = max(ligne_depart["wgs84Lat"], ligne_arrivee["wgs84Lat"]) + MARGE_BBOX_DEG
ouest = min(ligne_depart["wgs84Lon"], ligne_arrivee["wgs84Lon"]) - MARGE_BBOX_DEG
est = max(ligne_depart["wgs84Lon"], ligne_arrivee["wgs84Lon"]) + MARGE_BBOX_DEG

data_osm = requete_overpass(f'[out:json][timeout:60];way["railway"="rail"]({sud},{ouest},{nord},{est});out geom;')

lignes_osm, service_osm = [], []
for el in data_osm["elements"]:
    if el["type"] != "way" or "geometry" not in el:
        continue
    coords = [(p["lon"], p["lat"]) for p in el["geometry"]]
    if len(coords) < 2:
        continue
    lignes_osm.append(LineString(coords))
    service_osm.append(el.get("tags", {}).get("service"))

voies = gpd.GeoDataFrame({"service": service_osm}, geometry=lignes_osm, crs="EPSG:4326").to_crs("EPSG:2154")
voies_principales = voies[voies["service"].isna()]  # exclut triages/épis/raccordements
if voies_principales.empty:
    raise RuntimeError("Aucune voie ferrée OSM trouvée dans la zone du corridor (élargir MARGE_BBOX_DEG ?).")

# unary_union « node » le réseau aux intersections (nécessaire : les tronçons
# OSM ne sont pas systématiquement coupés à chaque croisement).
troncons = unary_union(list(voies_principales.geometry))
troncons = list(troncons.geoms) if hasattr(troncons, "geoms") else [troncons]


def arrondir(coord, precision=0.5):
    return (round(coord[0] / precision) * precision, round(coord[1] / precision) * precision)


reseau = nx.Graph()
for troncon in troncons:
    points = list(troncon.coords)
    for a, b in zip(points[:-1], points[1:]):
        na, nb = arrondir(a), arrondir(b)
        longueur = Point(a).distance(Point(b))
        if reseau.has_edge(na, nb) and reseau[na][nb]["length"] <= longueur:
            continue
        reseau.add_edge(na, nb, length=longueur)

gares_2154 = gares.to_crs("EPSG:2154")
point_depart = gares_2154.loc[gares["nomGare"] == GARE_DEPART, "geometry"].iloc[0]
point_arrivee = gares_2154.loc[gares["nomGare"] == GARE_ARRIVEE, "geometry"].iloc[0]
distance_vol_oiseau_km = point_depart.distance(point_arrivee) / 1000

noeuds = list(reseau.nodes())
points_noeuds = gpd.GeoSeries([Point(n) for n in noeuds], crs="EPSG:2154")


def noeud_le_plus_proche(point):
    distances = points_noeuds.distance(point)
    idx = distances.idxmin()
    return noeuds[idx]


noeud_depart = noeud_le_plus_proche(point_depart)
noeud_arrivee = noeud_le_plus_proche(point_arrivee)
if not nx.has_path(reseau, noeud_depart, noeud_arrivee):
    if distance_vol_oiseau_km > SEUIL_DISTANCE_ALERTE_KM:
        raise RuntimeError(
            f"{GARE_DEPART} et {GARE_ARRIVEE} sont à {distance_vol_oiseau_km:.0f} km à vol d'oiseau "
            f"(> {SEUIL_DISTANCE_ALERTE_KM} km) et aucune voie ferroviaire continue ne les relie dans les "
            "données OpenStreetMap récupérées : vérifiez qu'il existe bien une ligne directe entre ces "
            "deux gares."
        )
    raise RuntimeError(
        f"Aucun chemin ferré continu trouvé entre {GARE_DEPART} et {GARE_ARRIVEE} dans les données OSM "
        "récupérées (élargir MARGE_BBOX_DEG ?)."
    )
chemin_noeuds = nx.shortest_path(reseau, noeud_depart, noeud_arrivee, weight="length")
ligne_voie = LineString(chemin_noeuds)
distance_directe_km = ligne_voie.length / 1000
zone_voie = ligne_voie.buffer(TAMPON_VOIE_M)

gares_2154["sur_la_voie"] = gares_2154.geometry.within(zone_voie)
gares_dans_tolerance = gares_2154[gares_2154["sur_la_voie"]].copy()
gares_dans_tolerance["distance_depart_km"] = gares_dans_tolerance.geometry.apply(
    lambda g: ligne_voie.project(g) / 1000
)
gares_dans_tolerance = gares_dans_tolerance.sort_values("distance_depart_km")

# Une gare de corridor par commune : quand une agglomération en a
# plusieurs, on ne garde que la plus fréquentée.
frequentation_recente = app.load_frequentation()[["codeUic", "voyageurs"]]
gares_dans_tolerance = gares_dans_tolerance.merge(frequentation_recente, on="codeUic", how="left")
gares_dans_tolerance["voyageurs"] = gares_dans_tolerance["voyageurs"].fillna(0)
gares_corridor = (
    gares_dans_tolerance.sort_values("voyageurs", ascending=False)
    .drop_duplicates(subset="inseeCommune", keep="first")
    .sort_values("distance_depart_km")
    .reset_index(drop=True)
)

# Aire d'influence Géofer 10 min en voiture de chaque gare du corridor.
isochrones = gpd.read_file(f"{GEOFER_DIR}/iso_10min_voiture.geojson").to_crs("EPSG:2154")
isochrones_corridor = isochrones[isochrones["code_uic"].isin(gares_corridor["codeUic"])].merge(
    gares_corridor[["codeUic", "inseeCommune", "nomGare", "distance_depart_km", "geometry"]].rename(
        columns={"geometry": "point_gare", "inseeCommune": "gare_corridor_inseeCommune", "nomGare": "gare_corridor"}
    ),
    left_on="code_uic", right_on="codeUic",
)

# Communes réellement recoupées par ces zones (pas seulement celles qui ont une gare).
departements_corridor = gares_corridor["inseeDepartement"].unique()
communes = []
for dept in departements_corridor:
    r = requests.get(
        "https://geo.api.gouv.fr/communes",
        params={"codeDepartement": dept, "geometry": "contour", "format": "geojson", "fields": "nom,code"},
        timeout=30,
    )
    r.raise_for_status()
    communes.append(gpd.GeoDataFrame.from_features(r.json()["features"], crs="EPSG:4326"))
communes = pd.concat(communes, ignore_index=True)
# Union de toutes les communes des départements traversés (pas juste
# l'aire d'influence des gares) : sert à cadrer la couche carreaux
# population sur la carte (cellule 6), en WGS84 comme carreaux_france.
departements_geom = communes.geometry.union_all()
communes = communes.to_crs("EPSG:2154")

# Rattachement de chaque commune à la gare du corridor la plus proche quand
# elle recoupe plusieurs zones d'influence (rare, mais les zones voisines
# peuvent border la même commune).
communes_influence = []
for _, gare_row in isochrones_corridor.iterrows():
    recoupe = communes[communes.intersects(gare_row["geometry"])].copy()
    recoupe["gare_corridor"] = gare_row["gare_corridor"]
    recoupe["gare_corridor_inseeCommune"] = gare_row["gare_corridor_inseeCommune"]
    recoupe["gare_corridor_position_km"] = gare_row["distance_depart_km"]
    recoupe["distance_a_la_gare_corridor_km"] = recoupe.geometry.centroid.distance(gare_row["point_gare"]) / 1000
    communes_influence.append(recoupe)

communes_influence = pd.concat(communes_influence, ignore_index=True)
# La commune d'une gare est toujours dans sa PROPRE isochrone (le point de
# la gare y est), mais deux gares de corridor très proches (ex. Bordeaux/
# Cenon) peuvent faire gagner la gare voisine sur le seul critère de
# distance au centroïde — la commune d'une gare doit toujours lui rester
# rattachée, sans quoi ses propres flux domicile-travail/études seraient
# comptés sur la mauvaise gare (jusqu'à un faux "Cenon -> Cenon" après
# fusion, deux communes distinctes devenant le même nœud par erreur).
communes_influence["propre_gare"] = communes_influence["code"] == communes_influence["gare_corridor_inseeCommune"]
communes_influence = communes_influence.sort_values(
    ["propre_gare", "distance_a_la_gare_corridor_km"], ascending=[False, True]
).drop_duplicates(subset="code", keep="first").drop(columns="propre_gare")
communes_influence = communes_influence.rename(columns={"code": "inseeCommune", "nom": "nomCommune"})
communes_influence = communes_influence.sort_values("gare_corridor_position_km")[
    ["gare_corridor_position_km", "gare_corridor", "gare_corridor_inseeCommune",
     "inseeCommune", "nomCommune", "distance_a_la_gare_corridor_km"]
].reset_index(drop=True)

# Fichiers caches intermédiaires (réutilisables sans tout recalculer).
gares_corridor.drop(columns="geometry").to_csv(os.path.join(dossier_corridor, "gares_corridor.csv"), index=False)
communes_influence.to_csv(os.path.join(dossier_corridor, "communes_influence.csv"), index=False)
communes_influence


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-09-12 17:58:28.858 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager


2026-09-12 17:58:28.859 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager


2026-09-12 17:58:28.859 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager


2026-09-12 17:58:28.860 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager


2026-09-12 17:58:28.860 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager


...

2026-09-12 17:58:28.861 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager


2026-09-12 17:58:28.862 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager


2026-09-12 17:58:31.309 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager


2026-09-12 17:58:31.310 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


,gare_corridor_position_km,gare_corridor,gare_corridor_inseeCommune,inseeCommune,nomCommune,distance_a_la_gare_corridor_km
0,0.000000,Reims,51454,51055,Bétheny,4.828291
1,0.000000,Reims,51454,51418,Ormes,5.326310
2,0.000000,Reims,51454,51657,Vrigny,8.150566
3,0.000000,Reims,51454,51282,Gueux,8.558862
4,0.000000,Reims,51454,51454,Reims,1.401044
5,0.000000,Reims,51454,51518,Saint-Thierry,5.563401
6,0.000000,Reims,51454,51365,Les Mesneux,6.189856
7,0.000000,Reims,51454,51569,Thillois,5.759059
8,0.000000,Reims,51454,51474,Saint-Brice-Courcelles,2.802024
9,0.000000,Reims,51454,51573,Tinqueux,2.746155


In [2]:
# Flux domicile-travail / domicile-études entre les gares du corridor : les
# communes de l'aire d'influence de chaque gare (cellule précédente) sont
# fusionnées sur cette gare avant le calcul des flux — même logique que la
# fusion des arrondissements Paris/Lyon/Marseille dans app.load_flux.
fusion_codes = communes_influence.set_index("inseeCommune")["gare_corridor_inseeCommune"]
fusion_labels = communes_influence.set_index("inseeCommune")["gare_corridor"]

def fusionner_sur_gare(df):
    df = df.copy()
    for col_code, col_label in [("origine", "label_origine"), ("destination", "label_destination")]:
        df[col_code] = df[col_code].map(lambda c: fusion_codes.get(c, c))
        df[col_label] = df[col_code].map(fusion_labels).fillna(df[col_label])
    return df.groupby(
        ["origine", "label_origine", "destination", "label_destination"], as_index=False
    )["flux"].sum()

gares_corridor_codes = set(communes_influence["gare_corridor_inseeCommune"])

flux_corridor = pd.concat(
    [
        fusionner_sur_gare(app.load_flux(theme))[
            lambda df: df["origine"].isin(gares_corridor_codes) & df["destination"].isin(gares_corridor_codes)
        ].assign(theme=theme)
        for theme in ["travail", "etudes"]
    ],
    ignore_index=True,
)
# Flux "internes" (origine == destination après fusion, ex. reste dans la
# même aire d'influence de gare) : pas une liaison entre deux points du
# corridor, on les retire.
flux_corridor = flux_corridor[flux_corridor["origine"] != flux_corridor["destination"]]
flux_corridor = flux_corridor.sort_values(["theme", "flux"], ascending=[True, False]).reset_index(drop=True)

flux_corridor.to_csv(os.path.join(dossier_corridor, "flux_corridor.csv"), index=False)
flux_corridor


2026-09-12 17:58:34.601 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


2026-09-12 17:58:39.903 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


,origine,label_origine,destination,label_destination,flux,theme
0,51584,Trois Puits,51454,Reims,1499.888022,etudes
1,51230,Epernay,51454,Reims,504.243880,etudes
2,51454,Reims,51584,Trois Puits,314.135604,etudes
3,51028,Avenay,51230,Epernay,287.261217,etudes
4,51375,Montbré,51454,Reims,276.287127,etudes
...,...,...,...,...,...,...
77,51230,Epernay,51266,Germaine,7.427606,travail
78,51584,Trois Puits,51266,Germaine,7.375598,travail
79,51028,Avenay,51266,Germaine,5.238465,travail
80,51028,Avenay,51375,Montbré,4.772727,travail


In [3]:
# Fréquentation annuelle de chaque gare du corridor (historique par année,
# depuis Data_SNCF/frequentation_gares.csv — pas geofer_gares.csv, qui n'a
# qu'une colonne par année plutôt qu'un vrai historique long). Une seule
# gare par commune (cf. cellule 1 : la plus fréquentée quand une
# agglomération en a plusieurs).
SNCF_DIR = "Data_SNCF"

frequentation = pd.read_csv(f"{SNCF_DIR}/frequentation_gares.csv", dtype={"codeUic": str})
frequentation_corridor = frequentation[frequentation["codeUic"].isin(gares_corridor["codeUic"])].merge(
    gares_corridor[["codeUic", "distance_depart_km"]], on="codeUic",
).sort_values(["distance_depart_km", "annee"]).reset_index(drop=True)

chemin_csv = os.path.join(dossier_corridor, "frequentation_gares.csv")
frequentation_corridor.to_csv(chemin_csv, index=False)
print(f"{len(frequentation_corridor)} lignes écrites dans {chemin_csv}")
frequentation_corridor


80 lignes écrites dans Output/Reims_Epernay_corridor/frequentation_gares.csv


,codeUic,nomGare,nomCommune,voyageurs,annee,distance_depart_km
0,87171009,Reims,REIMS,3884021,2015,0.000000
1,87171009,Reims,REIMS,3751481,2016,0.000000
2,87171009,Reims,REIMS,3966061,2017,0.000000
3,87171009,Reims,REIMS,3643133,2018,0.000000
4,87171009,Reims,REIMS,3820119,2019,0.000000
...,...,...,...,...,...,...
75,87171553,Epernay,EPERNAY,479008,2020,30.465517
76,87171553,Epernay,EPERNAY,592899,2021,30.465517
77,87171553,Epernay,EPERNAY,835958,2022,30.465517
78,87171553,Epernay,EPERNAY,973775,2023,30.465517


In [4]:
# Population dans l'aire d'influence 10 min en voiture de chaque gare du
# corridor, à partir des carreaux INSEE 200x200 (Filosofi 2019) déjà
# utilisés par l'appli (app.load_carreaux_france). Partage de l'aire entre
# gares voisines par la médiane (diagramme de Voronoï, en mètres) : sans
# ça, une zone où deux isochrones se recoupent serait comptée deux fois —
# la somme par gare doit rester cohérente avec la population totale sans
# double compte (cf. cellule 5).
from shapely.geometry import MultiPoint
from shapely.ops import voronoi_diagram

carreaux_france = app.load_carreaux_france()

isochrones_corridor_completes = isochrones[isochrones["code_uic"].isin(gares_corridor["codeUic"])].merge(
    gares_corridor[["codeUic", "nomGare", "nomCommune", "distance_depart_km"]],
    left_on="code_uic", right_on="codeUic",
)  # encore en EPSG:2154 (celui de `isochrones`), nécessaire pour un Voronoï correct en mètres

points_gares = MultiPoint(list(gares_corridor.geometry))
cellules_voronoi = list(voronoi_diagram(points_gares).geoms)
cellule_par_gare = {
    gare["codeUic"]: next(c for c in cellules_voronoi if gare.geometry.within(c))
    for _, gare in gares_corridor.iterrows()
}
isochrones_corridor_completes["geometry"] = isochrones_corridor_completes.apply(
    lambda row: row["geometry"].intersection(cellule_par_gare[row["code_uic"]]), axis=1
)
isochrones_corridor_completes = isochrones_corridor_completes.to_crs("EPSG:4326")  # carreaux_france est en WGS84


def population_dans_isochrone(polygon):
    if polygon.is_empty:
        return 0
    minx, miny, maxx, maxy = polygon.bounds
    sous = carreaux_france.cx[minx:maxx, miny:maxy]
    if sous.empty:
        return 0
    return sous[sous.intersects(polygon)]["pop"].sum()


isochrones_corridor_completes["population_10min_voiture"] = isochrones_corridor_completes["geometry"].apply(
    population_dans_isochrone
)

population_corridor = isochrones_corridor_completes[
    ["distance_depart_km", "nomGare", "nomCommune", "population_10min_voiture"]
].sort_values("distance_depart_km").reset_index(drop=True)

population_corridor.to_csv(os.path.join(dossier_corridor, "population_gares.csv"), index=False)
population_corridor


2026-09-12 17:58:41.794 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


,distance_depart_km,nomGare,nomCommune,population_10min_voiture
0,0.000000,Reims,REIMS,64658.0
1,6.618474,Trois Puits,TROIS-PUITS,4761.5
2,8.013712,Montbré,MONTBRE,1227.0
3,11.126792,Rilly la Montagne,RILLY-LA-MONTAGNE,2542.5
4,15.655470,Germaine,GERMAINE,789.0
5,23.430837,Avenay,AVENAY-VAL-D'OR,3338.0
6,27.174874,Ay,AY-CHAMPAGNE,3969.5
7,30.465517,Epernay,EPERNAY,23493.5


In [5]:
# Synthèse du corridor : gares intermédiaires, population desservie (par
# gare + totale, sans double comptage grâce au partage Voronoï), flux
# domicile-travail/domicile-études totaux, et principales origines-
# destinations sur le cumul des deux.
from IPython.display import display


def formater_colonnes_entieres(df: pd.DataFrame, colonnes: list) -> pd.DataFrame:
    """Copie d'affichage avec ces colonnes en entiers, espace comme
    séparateur de milliers (ex. "25 000")."""
    df = df.copy()
    for colonne in colonnes:
        df[colonne] = df[colonne].map(lambda v: f"{v:,.0f}".replace(",", " "))
    return df


def formater_colonnes_distance(df: pd.DataFrame, colonnes: list) -> pd.DataFrame:
    """Copie d'affichage avec ces colonnes (km) à une décimale."""
    df = df.copy()
    for colonne in colonnes:
        df[colonne] = df[colonne].map(lambda v: f"{v:.1f}")
    return df

NB_TOP_OD = 10  # paramétrable

print(f"=== Corridor {GARE_DEPART} - {GARE_ARRIVEE} ({distance_directe_km:.1f} km le long de la voie) ===\n")

gares_intermediaires_liste = gares_corridor[
    ~gares_corridor["nomGare"].isin([GARE_DEPART, GARE_ARRIVEE])
]["nomGare"].tolist()
print(f"Gares intermédiaires ({len(gares_intermediaires_liste)}) :")
print(", ".join(gares_intermediaires_liste))

# Zone unique (union) : la vraie population couverte, pour vérifier que la
# somme par gare (déjà partagée par Voronoï) ne s'en écarte pas trop —
# un léger écart reste possible aux confins de deux zones voisines.
aire_influence_totale = isochrones_corridor_completes.union_all()
minx, miny, maxx, maxy = aire_influence_totale.bounds
sous = carreaux_france.cx[minx:maxx, miny:maxy]
population_totale_dedupliquee = sous[sous.intersects(aire_influence_totale)]["pop"].sum()

print("\nPopulation desservie (10 min voiture) :")
print(f"  - somme par gare (partage Voronoï) : {int(population_corridor['population_10min_voiture'].sum()):,}".replace(",", " "))
print(f"  - zone unique (référence) : {int(population_totale_dedupliquee):,}".replace(",", " "))
print("\nPopulation par gare :")
population_affichee = formater_colonnes_entieres(population_corridor, ["population_10min_voiture"])
population_affichee = formater_colonnes_distance(population_affichee, ["distance_depart_km"])
display(population_affichee)

flux_travail_total = flux_corridor.loc[flux_corridor["theme"] == "travail", "flux"].sum()
flux_etudes_total = flux_corridor.loc[flux_corridor["theme"] == "etudes", "flux"].sum()
print(f"\nSomme domicile-travail dans le corridor : {flux_travail_total:,.0f}".replace(",", " "))
print(f"Somme domicile-études dans le corridor : {flux_etudes_total:,.0f}".replace(",", " "))

flux_cumul_od = flux_corridor.groupby(
    ["origine", "label_origine", "destination", "label_destination"], as_index=False
)["flux"].sum().sort_values("flux", ascending=False)
top_od = flux_cumul_od.head(NB_TOP_OD).reset_index(drop=True)
print(f"\nTop {NB_TOP_OD} origines-destinations (cumul domicile-travail + domicile-études) :")
display(formater_colonnes_entieres(top_od, ["flux"]))


=== Corridor Reims - Epernay (30.5 km le long de la voie) ===

Gares intermédiaires (6) :
Trois Puits, Montbré, Rilly la Montagne, Germaine, Avenay, Ay

Population desservie (10 min voiture) :
  - somme par gare (partage Voronoï) : 104 779
  - zone unique (référence) : 104 385

Population par gare :


,distance_depart_km,nomGare,nomCommune,population_10min_voiture
0,0.0,Reims,REIMS,64 658
1,6.6,Trois Puits,TROIS-PUITS,4 762
2,8.0,Montbré,MONTBRE,1 227
3,11.1,Rilly la Montagne,RILLY-LA-MONTAGNE,2 542
4,15.7,Germaine,GERMAINE,789
5,23.4,Avenay,AVENAY-VAL-D'OR,3 338
6,27.2,Ay,AY-CHAMPAGNE,3 970
7,30.5,Epernay,EPERNAY,23 494



Somme domicile-travail dans le corridor : 16 748
Somme domicile-études dans le corridor : 4 412

Top 10 origines-destinations (cumul domicile-travail + domicile-études) :


,origine,label_origine,destination,label_destination,flux
0,51584,Trois Puits,51454,Reims,4 620
1,51454,Reims,51584,Trois Puits,4 448
2,51454,Reims,51230,Epernay,1 762
3,51230,Epernay,51454,Reims,1 585
4,51028,Avenay,51230,Epernay,1 033
5,51461,Rilly la Montagne,51454,Reims,910
6,51375,Montbré,51454,Reims,865
7,51030,Ay,51230,Epernay,809
8,51028,Avenay,51030,Ay,416
9,51028,Avenay,51454,Reims,380


In [6]:
# Charge cumulée par tronçon (segment entre deux gares consécutives du
# corridor) : somme des flux domicile-travail/étude dont le trajet
# traverse ce tronçon — même principe de profil de charge que
# creer_carte_troncons dans github.com/antoinechevre/GTFS_analysis_fr,
# appliqué aux flux INSEE plutôt qu'aux passages GTFS (un trajet "traverse"
# un tronçon si son origine et sa destination encadrent les deux gares du
# tronçon, tous sens confondus).
positions_gares = gares_corridor.set_index("inseeCommune")["distance_depart_km"]
gares_ordonnees = gares_corridor.sort_values("distance_depart_km").reset_index(drop=True)


def calculer_charge_troncons(df_flux: pd.DataFrame) -> pd.DataFrame:
    lignes = []
    for i in range(len(gares_ordonnees) - 1):
        gare_a, gare_b = gares_ordonnees.iloc[i], gares_ordonnees.iloc[i + 1]
        pos_a, pos_b = gare_a["distance_depart_km"], gare_b["distance_depart_km"]
        pos_origine = df_flux["origine"].map(positions_gares)
        pos_destination = df_flux["destination"].map(positions_gares)
        borne_min = pd.concat([pos_origine, pos_destination], axis=1).min(axis=1)
        borne_max = pd.concat([pos_origine, pos_destination], axis=1).max(axis=1)
        traverse = (borne_min <= pos_a) & (borne_max >= pos_b)
        lignes.append({
            "troncon": f"{gare_a['nomGare']} — {gare_b['nomGare']}",
            "position_depart_km": pos_a,
            "position_arrivee_km": pos_b,
            "charge": df_flux.loc[traverse, "flux"].sum(),
        })
    return pd.DataFrame(lignes)


flux_cumul_od = flux_corridor.groupby(
    ["origine", "label_origine", "destination", "label_destination"], as_index=False
)["flux"].sum()

charge_troncons_travail = calculer_charge_troncons(flux_corridor[flux_corridor["theme"] == "travail"])
charge_troncons_etudes = calculer_charge_troncons(flux_corridor[flux_corridor["theme"] == "etudes"])
charge_troncons_cumul = calculer_charge_troncons(flux_cumul_od)

charge_troncons_cumul.to_csv(os.path.join(dossier_corridor, "charge_troncons.csv"), index=False)
charge_affichee = formater_colonnes_entieres(charge_troncons_cumul, ["charge"])
charge_affichee = formater_colonnes_distance(charge_affichee, ["position_depart_km", "position_arrivee_km"])
charge_affichee


,troncon,position_depart_km,position_arrivee_km,charge
0,Reims — Trois Puits,0.0,6.6,16 210
1,Trois Puits — Montbré,6.6,8.0,8 042
2,Montbré — Rilly la Montagne,8.0,11.1,7 124
3,Rilly la Montagne — Germaine,11.1,15.7,5 607
4,Germaine — Avenay,15.7,23.4,5 410
5,Avenay — Ay,23.4,27.2,6 690
6,Ay — Epernay,27.2,30.5,6 674


In [7]:
# Carte HTML du corridor, couches sélectionnables via le LayerControl natif
# (comme dans app.py) : 3 fonds de carte (OpenStreetMap, CartoDB Positron,
# CartoDB Dark Matter), carreaux INSEE 200x200 (population), les 3
# isochrones Géofer (10 min voiture/vélo, 15 min à pied), fréquentation et
# offre des gares, et les flèches domicile-études / domicile-travail /
# cumulées. Réutilise les briques de rendu de app.py pour rester cohérent
# visuellement avec l'appli.
import math

import folium

# --- Carreaux INSEE 200x200 (population), sur tous les départements traversés ---
minx_dept, miny_dept, maxx_dept, maxy_dept = departements_geom.bounds
carreaux_corridor = carreaux_france.cx[minx_dept:maxx_dept, miny_dept:maxy_dept]
carreaux_corridor = carreaux_corridor[carreaux_corridor.intersects(departements_geom)]

# Le cadrage initial de la carte reste celui du corridor lui-même (pas les
# départements entiers, sinon la ligne devient minuscule à l'écran).
minx, miny, maxx, maxy = aire_influence_totale.bounds
centre_carte = [(miny + maxy) / 2, (minx + maxx) / 2]
m = folium.Map(location=centre_carte, tiles=None, prefer_canvas=True, control_scale=True)
folium.TileLayer("OpenStreetMap", name="OpenStreetMap", cross_origin=True).add_to(m)
folium.TileLayer(
    app.carto_tile_url("light_all"), name="CartoDB Positron", attr=app.CARTO_ATTR, cross_origin=True,
).add_to(m)
folium.TileLayer(
    app.carto_tile_url("dark_all"), name="CartoDB Dark Matter", attr=app.CARTO_ATTR, cross_origin=True,
).add_to(m)

if not carreaux_corridor.empty:
    vmin = float(carreaux_corridor["pop"].min())
    vmax = float(carreaux_corridor["pop"].max())
    colormap = folium.LinearColormap(
        colors=app.CARREAUX_COLOR_SCALE, vmin=vmin, vmax=vmax, caption="Population (carreau 200m)"
    )

    def style_carreau(feature, lo=vmin, hi=vmax):
        value = feature["properties"]["pop"]
        if hi > lo:
            frac = ((value - lo) / (hi - lo)) ** app.CARREAUX_COLOR_GAMMA
            value = lo + frac * (hi - lo)
        return {"fillColor": colormap(value), "color": "#581012", "weight": 0, "fillOpacity": 1.0}

    folium.GeoJson(
        carreaux_corridor[["pop", "geometry"]],
        name="Carreaux INSEE 200m (population)",
        style_function=style_carreau,
        tooltip=folium.GeoJsonTooltip(fields=["pop"], aliases=["Population"], localize=True),
    ).add_to(m)
    colormap.add_to(m)

# --- Isochrones Géofer, gares du corridor uniquement ---
codes_uic_corridor = set(gares_corridor["codeUic"])
for mode, (path, couleur) in app.ISOCHRONE_FILES.items():
    gdf = gpd.read_file(path)
    gdf = gdf[gdf["code_uic"].isin(codes_uic_corridor)]
    if gdf.empty:
        continue
    folium.GeoJson(
        gdf, name=mode,
        style_function=lambda feature, c=couleur: {
            "color": c, "weight": 2, "fill": True, "fillColor": c, "fillOpacity": 0.3,
        },
    ).add_to(m)

# --- Fréquentation des gares (bulles proportionnelles au max du corridor) ---
frequentation = app.load_frequentation()
frequentation_corridor = gares_corridor[["codeUic", "wgs84Lat", "wgs84Lon"]].merge(
    frequentation, on="codeUic", how="inner"
)
if not frequentation_corridor.empty:
    freq_max = frequentation_corridor["voyageurs"].max()
    freq_layer = folium.FeatureGroup(name=f"Fréquentation {app.FREQUENTATION_ANNEE} (voyageurs/an)")
    for _, gare_freq in frequentation_corridor.iterrows():
        rayon = app.FREQUENTATION_MIN_RADIUS_PX + (
            app.FREQUENTATION_MAX_RADIUS_PX - app.FREQUENTATION_MIN_RADIUS_PX
        ) * math.sqrt(gare_freq["voyageurs"] / freq_max)
        folium.Marker(
            [gare_freq["wgs84Lat"], gare_freq["wgs84Lon"]],
            icon=folium.DivIcon(
                html=app.frequentation_bubble_svg(rayon), icon_size=(rayon * 2, rayon * 2), icon_anchor=(rayon, rayon)
            ),
            tooltip=(
                f"{gare_freq['nomGare']} — {int(gare_freq['voyageurs']):,} voyageurs/an "
                f"({app.FREQUENTATION_ANNEE})"
            ).replace(",", " "),
        ).add_to(freq_layer)
    freq_layer.add_to(m)

# --- Offre 2026 (camemberts TER/Intercités/TGV, proportionnels au max du corridor) ---
offre = app.load_offre_2026()
offre_corridor = gares_corridor[["codeUic", "wgs84Lat", "wgs84Lon"]].merge(
    offre, on="codeUic", how="inner"
)
if not offre_corridor.empty:
    offre_max = offre_corridor["totalClasse"].max()
    offre_layer = folium.FeatureGroup(name="Offre 2026 (TER / Intercités / TGV)")
    for _, gare_offre in offre_corridor.iterrows():
        rayon = app.OFFRE_MIN_RADIUS_PX + (app.OFFRE_MAX_RADIUS_PX - app.OFFRE_MIN_RADIUS_PX) * math.sqrt(
            gare_offre["totalClasse"] / offre_max
        )
        folium.Marker(
            [gare_offre["wgs84Lat"], gare_offre["wgs84Lon"]],
            icon=folium.DivIcon(
                html=app.offre_pie_svg(gare_offre, rayon), icon_size=(rayon * 2, rayon * 2), icon_anchor=(rayon, rayon)
            ),
            tooltip=app.offre_popup(gare_offre),
        ).add_to(offre_layer)
    offre_layer.add_to(m)

# --- Flèches domicile-études / domicile-travail / cumulées, entre gares du corridor ---
# SEUIL_MIN_FLUX : sous ce nombre de personnes, la flèche n'est pas dessinée
# (paramétrable, pour ne pas noyer la carte de flux marginaux). Courbure
# différente par thème : sépare visuellement les flèches quand plusieurs
# couches se recoupent entre les deux mêmes gares.
SEUIL_MIN_FLUX = 10  # domicile-travail / domicile-études : masque les flux < 10 personnes
COURBURE_PAR_THEME = {"travail": 0.18, "etudes": 0.30, "cumul": 0.24}

coords_gare_corridor = gares_corridor.set_index("inseeCommune")[["wgs84Lat", "wgs84Lon"]]
flux_cumul_od = flux_corridor.groupby(
    ["origine", "label_origine", "destination", "label_destination"], as_index=False
)["flux"].sum()

flux_layers = {
    "travail": (flux_corridor[flux_corridor["theme"] == "travail"], "#1f78b4", "Flèches domicile-travail"),
    "etudes": (flux_corridor[flux_corridor["theme"] == "etudes"], "#e6550d", "Flèches domicile-études"),
    "cumul": (flux_cumul_od, "#1a9850", "Flèches cumulées domicile-travail + domicile-études"),
}
for theme, (df_theme, couleur, nom_calque) in flux_layers.items():
    df_theme = df_theme[df_theme["flux"] >= SEUIL_MIN_FLUX]
    if df_theme.empty:
        continue
    flux_max = df_theme["flux"].max()
    couche = folium.FeatureGroup(name=nom_calque)
    for _, flux in df_theme.iterrows():
        origine_latlon = tuple(coords_gare_corridor.loc[flux["origine"]])
        destination_latlon = tuple(coords_gare_corridor.loc[flux["destination"]])
        poids = app.FLUX_MIN_WEIGHT_PX + (app.FLUX_MAX_WEIGHT_PX - app.FLUX_MIN_WEIGHT_PX) * math.sqrt(
            flux["flux"] / flux_max
        )
        courbe = app.bezier_arc(origine_latlon, destination_latlon, courbure=COURBURE_PAR_THEME[theme])
        folium.PolyLine(
            courbe, color=couleur, weight=poids, opacity=0.75,
            tooltip=f"{flux['label_origine']} → {flux['label_destination']} — {flux['flux']:.0f} personnes",
        ).add_to(couche)
        angle = app.angle_entre_points(courbe[-2], courbe[-1])
        taille = poids + 8
        folium.Marker(
            courbe[-1],
            icon=folium.DivIcon(
                html=app.arrowhead_svg(angle, couleur, taille_px=int(taille)),
                icon_size=(taille, taille), icon_anchor=(taille / 2, taille / 2),
            ),
        ).add_to(couche)
    couche.add_to(m)

# --- Charge cumulée par tronçon (domicile-travail + domicile-études), sur le tracé réel de la voie ---
from shapely.ops import substring

if not charge_troncons_cumul.empty and charge_troncons_cumul["charge"].max() > 0:
    charge_min = charge_troncons_cumul["charge"].min()
    charge_max = charge_troncons_cumul["charge"].max()
    colormap_charge = folium.LinearColormap(
        colors=["#fee5d9", "#fcae91", "#fb6a4a", "#de2d26", "#a50f15"],
        vmin=charge_min, vmax=charge_max,
        caption="Charge cumulée par tronçon (domicile-travail + domicile-études)",
    )
    couche_charge = folium.FeatureGroup(name="Charge cumulée par tronçon")
    for _, troncon in charge_troncons_cumul.iterrows():
        segment = substring(ligne_voie, troncon["position_depart_km"] * 1000, troncon["position_arrivee_km"] * 1000)
        segment_wgs84 = gpd.GeoSeries([segment], crs="EPSG:2154").to_crs("EPSG:4326").iloc[0]
        coords = [(lat, lon) for lon, lat in segment_wgs84.coords]
        poids = (
            6 + (troncon["charge"] - charge_min) / (charge_max - charge_min) * 16
            if charge_max > charge_min else 8
        )
        folium.PolyLine(
            coords, color=colormap_charge(troncon["charge"]), weight=poids, opacity=0.85,
            tooltip=f"{troncon['troncon']} — {troncon['charge']:.0f} personnes (cumul)",
        ).add_to(couche_charge)
    couche_charge.add_to(m)
    colormap_charge.add_to(m)

# --- Repère des gares du corridor + cadrage ---
couche_gares = folium.FeatureGroup(name="Gares du corridor")
for _, gare in gares_corridor.iterrows():
    folium.CircleMarker(
        [gare["wgs84Lat"], gare["wgs84Lon"]], radius=4, color="#000", fill=True, fill_opacity=1,
        tooltip=gare["nomGare"],
    ).add_to(couche_gares)
couche_gares.add_to(m)

m.fit_bounds([[miny, minx], [maxy, maxx]])
folium.LayerControl(collapsed=False).add_to(m)

chemin_html = os.path.join(dossier_corridor, "carte_corridor.html")
m.save(chemin_html)
# Pas d'affichage inline de `m` : avec les carreaux INSEE à l'échelle du
# département (cellule 1), le HTML de la carte pèse une quinzaine de Mo —
# trop pour le rendu de sortie de cellule de la plupart des notebooks (fond
# de carte qui n'apparaît pas, widget qui ne charge pas). Le fichier
# complet, lui, s'ouvre normalement dans un navigateur.
print(f"Carte enregistrée dans {chemin_html} — à ouvrir directement dans un navigateur.")


2026-09-12 17:58:48.681 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager


2026-09-12 17:58:48.702 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


Carte enregistrée dans Output/Reims_Epernay_corridor/carte_corridor.html — à ouvrir directement dans un navigateur.
